# DalMax — Colab kernel diagnostic (run from VS Code)

Short **diagnostic** notebook. It does **not** run the campaign and trains nothing
beyond one tiny 1-round check. Every cell prints a `PASS` / `FAIL` / `SKIP` line
with the reason and never raises, so you can run all cells top to bottom and read
the final summary.

**How to connect**
1. In VS Code, install the **Google Colab** extension.
2. Open this notebook (`notebooks/colab_local_test.ipynb`).
3. Click **Select Kernel** -> **Colab** -> choose a **GPU** runtime.
4. Run all cells. A Google auth prompt may appear inside VS Code for the Drive mount.

**What is tested**: the kernel really is a Colab runtime, a GPU is visible, Google
Drive mounts (and the dataset is on it), the repo clones and `poetry install` works,
torch sees CUDA, dataset wiring works, and a tiny 1-round `trainer.py` run succeeds.

The full campaign lives in [`colab_runbook.ipynb`](colab_runbook.ipynb) (source of
truth: `COLAB_RUNBOOK.md`).

In [ ]:
import os

# Parameters -- same values as colab_runbook.ipynb.
DRIVE_ROOT = "/content/drive/MyDrive/UFMS/Pós-graduação/Doutorado/FINAL/PROJETO/DALMAX"
REPO_URL = "https://github.com/MarioCarvalhoBr/dalmax-deep-active-learning-python.git"
REPO_DIR = "/content/dalmax"
BRANCH = "main"
DATASET_ZIP_URL = ""  # optional, Fallback A only: shared-link zip of daninhas_full (gdown)
EXPECTED_FILES = 10193

os.environ["DRIVE_ROOT"] = DRIVE_ROOT

RESULTS = {}  # name -> (status, detail)


def record(name, status, detail=""):
    """Store and print one check result. status is PASS, FAIL or SKIP."""
    assert status in {"PASS", "FAIL", "SKIP"}, status
    RESULTS[name] = (status, detail)
    print(f"[{status}] {name}: {detail}")


print("DRIVE_ROOT:", DRIVE_ROOT)
print("REPO_DIR:  ", REPO_DIR, "| BRANCH:", BRANCH)

## 1. Where am I running?

In [ ]:
import importlib.util
import platform

try:
    print("platform:", platform.platform())
    print("cwd:", os.getcwd())
    has_content = os.path.isdir("/content")
    has_colab = importlib.util.find_spec("google.colab") is not None
    tag = os.environ.get("COLAB_RELEASE_TAG")
    gpu_env = os.environ.get("COLAB_GPU")
    print("/content present:", has_content, "| google.colab importable:", has_colab)
    print("COLAB_RELEASE_TAG:", tag, "| COLAB_GPU:", gpu_env)
    if has_content and has_colab:
        record("runtime", "PASS", "Colab runtime detected")
    else:
        record("runtime", "FAIL", "kernel is local, not Colab -- select a Colab kernel")
except Exception as exc:
    record("runtime", "FAIL", f"probe error: {exc!r}")

## 2. GPU and Python version

In [ ]:
import subprocess
import sys

try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=60,
    )
    gpu_ok = out.returncode == 0 and out.stdout.strip() != ""
    gpu_detail = out.stdout.strip() if gpu_ok else (out.stderr.strip() or "no GPU reported")
except Exception as exc:
    gpu_ok, gpu_detail = False, f"nvidia-smi unavailable: {exc!r}"
py = sys.version_info[:2]
py_ok = (3, 10) <= py <= (3, 12)
print("GPU:", gpu_detail)
print("Python:", sys.version.split()[0])
record("gpu", "PASS" if gpu_ok else "FAIL", gpu_detail)
record(
    "python_version", "PASS" if py_ok else "FAIL",
    f"{py[0]}.{py[1]}" + ("" if py_ok else " (torch==2.5.0 needs 3.10-3.12)"),
)

## 3. Google Drive mount probe

This is the main uncertainty being tested: whether `drive.mount` works from a
Colab kernel driven by VS Code. An auth prompt may appear in VS Code; there is
no timeout here, so answer it (or interrupt the cell if it hangs -- the summary
will then treat Drive as failed once you re-run the next cells).

In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
    data_dir = f"{DRIVE_ROOT}/DATA/daninhas_full"
    if not os.path.isdir(data_dir):
        record("drive", "FAIL", f"mounted, but {data_dir} not found (check DRIVE_ROOT)")
    else:
        n_files = sum(len(files) for _, _, files in os.walk(data_dir))
        print("files under daninhas_full:", n_files)
        if n_files >= EXPECTED_FILES:
            record("drive", "PASS", f"dataset on Drive, {n_files} files")
        else:
            record("drive", "FAIL", f"only {n_files} files (expected >= {EXPECTED_FILES}, incl. arquivos.txt)")
except Exception as exc:
    record("drive", "FAIL", f"{type(exc).__name__}: {exc}")

## 4. Fallback A -- Drive-free dataset (only runs if Drive FAILED)

Two ways to get `daninhas_full` without Drive:
- **Upload** `daninhas_full.zip` with the Colab extension's file upload into `/content/`.
- **gdown**: set `DATASET_ZIP_URL` in the Parameters cell to a shared-link zip.

The zip is extracted to `/content/daninhas_fallback/` and wired into the repo later.
**Caveat**: without Drive, results stay on the ephemeral runtime disk. Before the
runtime ends, run `zip -r results.zip results/campaign` and download it
(`from google.colab import files; files.download("results.zip")`, or download it
manually from the file browser).

In [ ]:
FALLBACK_DIR = "/content/daninhas_fallback"
try:
    if RESULTS.get("drive", ("", ""))[0] == "PASS":
        record("fallback_a", "SKIP", "Drive passed, fallback not needed")
    else:
        zip_path = "/content/daninhas_full.zip"
        if DATASET_ZIP_URL:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
            import gdown

            gdown.download(DATASET_ZIP_URL, zip_path, quiet=False, fuzzy=True)
        if not os.path.isfile(zip_path):
            record("fallback_a", "SKIP", "no DATASET_ZIP_URL and no uploaded /content/daninhas_full.zip")
        else:
            os.makedirs(FALLBACK_DIR, exist_ok=True)
            subprocess.run(["unzip", "-qo", zip_path, "-d", FALLBACK_DIR], check=True)
            n_files = sum(len(f) for _, _, f in os.walk(FALLBACK_DIR))
            status = "PASS" if n_files >= EXPECTED_FILES else "FAIL"
            record("fallback_a", status, f"{n_files} files extracted to {FALLBACK_DIR}")
except Exception as exc:
    record("fallback_a", "FAIL", f"{type(exc).__name__}: {exc}")

## 5. Repo clone/pull and Poetry install

In [ ]:
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "poetry"], check=True)
    inst = subprocess.run(["poetry", "install"], capture_output=True, text=True)
    print(inst.stdout[-1500:], inst.stderr[-1500:])
    if inst.returncode == 0:
        record("repo_poetry", "PASS", "repo ready, poetry install ok")
    else:
        record("repo_poetry", "FAIL", f"poetry install exit {inst.returncode}")
except Exception as exc:
    record("repo_poetry", "FAIL", f"{type(exc).__name__}: {exc}")

In [ ]:
try:
    probe = subprocess.run(
        ["poetry", "run", "python", "-c",
         "import torch, dalmax; print(torch.cuda.is_available(), torch.version.cuda)"],
        capture_output=True, text=True, cwd=REPO_DIR,
    )
    print(probe.stdout, probe.stderr[-800:])
    cuda_ok = probe.returncode == 0 and probe.stdout.split()[:1] == ["True"]
    record("torch_cuda", "PASS" if cuda_ok else "FAIL",
           probe.stdout.strip() or probe.stderr.strip()[-200:])
except Exception as exc:
    record("torch_cuda", "FAIL", f"{type(exc).__name__}: {exc}")

## 6. Data wiring

In [ ]:
import shutil

try:
    local_data = os.path.join(REPO_DIR, "DATA", "daninhas_full")
    if RESULTS.get("drive", ("", ""))[0] == "PASS":
        res = subprocess.run(["make", "colab-setup"], capture_output=True, text=True, cwd=REPO_DIR)
        print((res.stdout + res.stderr)[-1500:])
        record("data_wiring", "PASS" if res.returncode == 0 else "FAIL",
               f"make colab-setup exit {res.returncode}")
    elif RESULTS.get("fallback_a", ("", ""))[0] == "PASS":
        src = os.path.join(FALLBACK_DIR, "daninhas_full")
        src = src if os.path.isdir(src) else FALLBACK_DIR
        os.makedirs(os.path.join(REPO_DIR, "DATA"), exist_ok=True)
        if not os.path.isdir(local_data):
            shutil.copytree(src, local_data)
        n_files = sum(len(f) for _, _, f in os.walk(local_data))
        record("data_wiring", "PASS" if n_files >= EXPECTED_FILES else "FAIL",
               f"{n_files} files at {local_data}")
    else:
        record("data_wiring", "SKIP", "no dataset source (Drive failed, Fallback A not used)")
except Exception as exc:
    record("data_wiring", "FAIL", f"{type(exc).__name__}: {exc}")

## 7. Tiny GPU run (not the campaign)

In [ ]:
import glob
import json
import time

try:
    if RESULTS.get("data_wiring", ("", ""))[0] != "PASS":
        record("tiny_run", "SKIP", "dataset not wired")
    else:
        t0 = time.time()
        subprocess.run(["poetry", "run", "python", "scripts/make_micro_dataset.py"],
                       check=True, cwd=REPO_DIR)
        run = subprocess.run(
            ["poetry", "run", "python", "tools/trainer.py",
             "--params_json", "files_config/params_micro.json",
             "--dataset_name", "DANINHAS", "--strategy_name", "RandomSampling",
             "--n_init_labeled", "50", "--n_query", "20", "--n_round", "1",
             "--seed", "1", "--device", "cuda", "--dir_results", "results/colab_local_test/"],
            capture_output=True, text=True, cwd=REPO_DIR,
        )
        elapsed = time.time() - t0
        print((run.stdout + run.stderr)[-1500:])
        base = os.path.join(REPO_DIR, "results", "colab_local_test")
        found = glob.glob(os.path.join(base, "**", "results.json"), recursive=True)
        metas = glob.glob(os.path.join(base, "**", "run_metadata.json"), recursive=True)
        if metas:
            with open(metas[0], encoding="utf-8") as fh:
                meta = json.load(fh)
            env = meta.get("environment", {})
            gpus = env.get("gpus") or []
            print("environment.gpus[0]:", gpus[0] if gpus else None)
            print("determinism:", meta.get("determinism"))
        if run.returncode == 0 and found:
            record("tiny_run", "PASS", f"results.json written in {elapsed:.0f}s")
        else:
            record("tiny_run", "FAIL", f"exit {run.returncode}, results.json found: {bool(found)}")
except Exception as exc:
    record("tiny_run", "FAIL", f"{type(exc).__name__}: {exc}")

## 8. Summary and recommendation

In [ ]:
print(f"{'check':<16}{'status':<8}detail")
print("-" * 70)
for name, (status, detail) in RESULTS.items():
    print(f"{name:<16}{status:<8}{detail}")

def _st(name):
    return RESULTS.get(name, ("SKIP", ""))[0]

print()
if _st("runtime") == "FAIL" or _st("gpu") == "FAIL":
    print("RECOMMENDATION: the kernel is not a Colab GPU runtime. In VS Code: install the "
          "'Google Colab' extension, Select Kernel -> Colab -> choose a GPU runtime, then re-run.")
elif all(_st(n) in {"PASS", "SKIP"} for n in RESULTS) and _st("drive") == "PASS":
    print("RECOMMENDATION: you can run the campaign from VS Code: open notebooks/colab_runbook.ipynb "
          "with the same Colab kernel.")
elif _st("drive") == "FAIL":
    print("RECOMMENDATION: run the campaign in the Colab web UI (Drive works there), or use "
          "Fallback A and copy results manually before the runtime ends.")
else:
    print("RECOMMENDATION: fix the FAIL rows above (see details), then re-run this notebook.")